# Data EDA and Preparation for Stratified Splitting

In [1]:
import pandas as pd

In [2]:
mis_df = pd.read_csv('../data/annotations_1000.csv')

### Label Distribution

In [ ]:
# define function to explore distributions

def label_distribution(df, labels):
    '''
    Takes a dataframe and a list of column names corresponding to an annotation label.
    Calculates the distribution of each label. 
    '''
    for label in labels:
        label_dist = df[label].value_counts(normalize=True)
        print(label_dist)
   

In [4]:
labels = ['misinformation_label', 'opinion_label', 'annotator']
label_distribution(mis_df, labels)

misinformation_label
0    0.741
1    0.259
Name: proportion, dtype: float64
opinion_label
0    0.585
1    0.415
Name: proportion, dtype: float64
annotator
rachelle         0.258
shiao-li         0.232
nicole           0.205
jennifer         0.203
jasmine          0.059
Majority Vote    0.041
ai               0.002
Name: proportion, dtype: float64


### Text Length Distribution & trouble-shooting

In [5]:
mis_df['text_length'] = mis_df['text'].astype(str).str.len()
text_len_dist = mis_df['text_length'].describe()
print(text_len_dist)

count    1000.000000
mean      156.235000
std        73.433318
min        18.000000
25%       109.000000
50%       137.000000
75%       199.000000
max       558.000000
Name: text_length, dtype: float64


In [6]:
# check for texts that did not correctly transfer from json
mis_df[mis_df['text_length'] < 10]

,id,text,misinformation_label,opinion_label,all_caps,exclamation_marks,hedging,adjectives,unk,annotator,text_length


### Combinations in Dataset

In [ ]:
# Verify unique combinations of misinformation label, opinion label and annotator 
unique_combos = mis_df.groupby(labels).size().reset_index(name='count').sort_values('count', ascending=False)
unique_combos

,misinformation_label,opinion_label,annotator,count
2,0,0,jennifer,110
4,0,0,rachelle,101
3,0,0,nicole,96
5,0,0,shiao-li,83
11,0,1,rachelle,82
12,0,1,shiao-li,67
10,0,1,nicole,63
9,0,1,jennifer,52
18,1,0,shiao-li,51
17,1,0,rachelle,38


#### Combinations with Jasmine, Majority Vote & AI grouped together

In [8]:
# Verify unique combinations of misinformation label, opinion label and annotator, with Jasmine & Majority vote grouped
unique_combos_collapse_annot = mis_df.replace({'Majority Vote': 'Majority Vote, Jasmine, AI', 'jasmine': 'Majority Vote, Jasmine, AI', 'ai': 'Majority Vote, Jasmine, AI'}).groupby(labels).size().reset_index(name='count')#.sort_values('count', ascending=False)
unique_combos_collapse_annot

,misinformation_label,opinion_label,annotator,count
0,0,0,"Majority Vote, Jasmine, AI",54
1,0,0,jennifer,110
2,0,0,nicole,96
3,0,0,rachelle,101
4,0,0,shiao-li,83
5,0,1,"Majority Vote, Jasmine, AI",33
6,0,1,jennifer,52
7,0,1,nicole,63
8,0,1,rachelle,82
9,0,1,shiao-li,67


### Distribution of annotation labels & One-hot encoding

In [ ]:
# One-hot encode annotation labels 

mis_df['exclamation_marks_bin'] = mis_df['exclamation_marks'].apply(lambda x: 0 if x == '[]' else 1)
mis_df['all_caps_bin'] = mis_df['all_caps'].apply(lambda x: 0 if x == '[]' else 1)
mis_df['hedging_bin'] = mis_df['hedging'].apply(lambda x: 0 if x == '[]' else 1)
mis_df['adjectives_bin'] = mis_df['adjectives'].apply(lambda x: 0 if x == '[]' else 1)
mis_df['unk_bin'] = mis_df['unk'].apply(lambda x: 0 if x == '[]' else 1)

labels = ['misinformation_label', 'opinion_label', 'annotator', 'exclamation_marks_bin', 'all_caps_bin', 'hedging_bin', 'adjectives_bin', 'unk_bin']
label_distribution(mis_df, labels)

misinformation_label
0    0.741
1    0.259
Name: proportion, dtype: float64
opinion_label
0    0.585
1    0.415
Name: proportion, dtype: float64
annotator
rachelle         0.258
shiao-li         0.232
nicole           0.205
jennifer         0.203
jasmine          0.059
Majority Vote    0.041
ai               0.002
Name: proportion, dtype: float64
exclamation_marks_bin
0    0.867
1    0.133
Name: proportion, dtype: float64
all_caps_bin
0    0.921
1    0.079
Name: proportion, dtype: float64
hedging_bin
0    0.857
1    0.143
Name: proportion, dtype: float64
adjectives_bin
0    0.939
1    0.061
Name: proportion, dtype: float64
unk_bin
0    0.951
1    0.049
Name: proportion, dtype: float64


#### One-hot encode 4 main annotators

In [ ]:
# Given Majority Vote, Jasmine & AI only represent 10% annotations, not included in stratification

ann_ohe = pd.get_dummies(mis_df['annotator'], dtype=int)
ann_ohe = ann_ohe.drop(columns=['Majority Vote', 'jasmine', 'ai'])
ann_ohe

,jennifer,nicole,rachelle,shiao-li
0,0,0,0,0
1,0,0,0,0
2,0,0,0,0
3,0,0,0,0
4,0,0,0,0
...,...,...,...,...
995,0,0,0,1
996,0,0,0,1
997,0,0,0,1
998,0,0,0,1


In [12]:
# Concatenate mis_df and one-hot encoded annotator dataframe

mis_df = pd.concat([mis_df, ann_ohe], axis=1)
mis_df

,id,text,misinformation_label,opinion_label,all_caps,exclamation_marks,hedging,adjectives,unk,annotator,...,adjectives_bin,unk_bin,jennifer,nicole,rachelle,shiao-li,jennifer,nicole,rachelle,shiao-li
0,0,"No Food, No FEMA: Hurricane Michael’s Survivor...",0,0,[],[],[],[],[],Majority Vote,...,0,0,0,0,0,0,0,0,0,0
1,1,Would-be looter in Hurricane Michael-ravaged F...,0,1,[],"[['!', 315, 316]]",[],[],[],Majority Vote,...,0,0,0,0,0,0,0,0,0,0
2,2,His argument is correct. Hurricane Rita killed...,0,1,[],[],[],[],[],Majority Vote,...,0,0,0,0,0,0,0,0,0,0
3,3,im praying for all of my friends down in the C...,0,1,[],[],[],[],[],Majority Vote,...,0,0,0,0,0,0,0,0,0,0
4,4,To all my Texas streamers: PLEASE be safe if t...,0,1,"[['PLEASE', 27, 33]]","[['!', 71, 72]]",[],[],[],Majority Vote,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1098,Top link: Reddit's new content policy goes int...,1,0,[],[],[],"[['horrible', 60, 68]]",[],shiao-li,...,1,0,0,0,0,1,0,0,0,1
996,1099,@carolinagutierr grande twister!!!,1,0,[],"[['!!!', 31, 34]]",[],[],[],shiao-li,...,0,0,0,0,0,1,0,0,0,1
997,1100,"Dear @HRDMinistry ,\nJNU Is Not A House Of Lea...",1,1,[],[],[],"[['Intriguing', 155, 165]]",[],shiao-li,...,1,0,0,0,0,1,0,0,0,1
998,1103,The cool kids asked me if I wanted to hang out...,1,0,[],[],[],[],[],shiao-li,...,0,0,0,0,0,1,0,0,0,1


In [ ]:
# send dataframe to data folder in preparation for splitting

mis_df.to_csv('../data/mis_df_ohe.csv', index=False)